# Cell A: pin the model revision and run your demo on the real model (about 3 minutes)

In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/sarrazer24/evolutionary-prompt-optimization.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

from huggingface_hub import model_info
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SHA = model_info(MODEL_ID).sha
print("model:", MODEL_ID, "| pinned revision:", SHA)

env = dict(os.environ, HF_MODEL_REVISION=SHA)
r = subprocess.run([sys.executable, "-m", "scripts.member2_demo", "--real", "--n", "300"],
                   env=env, capture_output=True, text=True)
print(r.stdout[-12000:])
if r.returncode:
    print("STDERR:", r.stderr[-2000:])

Cloning into '/kaggle/working/repo'...


cb20496 04_model_check_and_demo file placeholder

model: Qwen/Qwen2.5-1.5B-Instruct | pinned revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "requested_revision": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
  "resolved_revision": null,
  "dtype": "float16",
  "device": "cuda:0",
  "gpu": "Tesla T4",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "inference_config": {
    "decoding": "greedy",
    "max_new_tokens": 8,
    "repetition_penalty": 1.0,
    "dtype": "float16",
    "user_template": "review-sentiment-v1"
  }
}
== 1. Classifying sample reviews with the first baseline prompt ==
  true=negative raw='negative'   parsed=negative  | "the film 's mid-to-low budget is betrayed by the surprisingl"
  true=negative raw='positive'   parsed=positive  | "it 'll probably be in video stores by christmas"
  true=negative raw='negative'   parsed=negative  | 'failure to construct a story with even a trace of dramatic i'
  true=negative

# Cell B: compare three candidate models on the same baselines

In [2]:
import gc, json, os, random, time, traceback
import pandas as pd
import torch
from huggingface_hub import model_info

from src.llm import LLMInterface, ModelConfig
from src.data.sst2_data import load_saved_splits, stratified_sample
from src.evaluation.evaluation import PredictionCache, evaluate_prompt
from src.prompts.baseline_prompts import get_baseline_prompts, get_baseline_prompt_texts
from src.prompts.initial_population import generate_initial_prompts
from src.prompts.operators import mutate_prompt

os.makedirs("/kaggle/working/outputs", exist_ok=True)
sample = stratified_sample(load_saved_splits("data/processed").opt, 500, seed=42)
CANDIDATES = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "microsoft/Phi-3.5-mini-instruct"]

rows, llm = [], None
for model_id in CANDIDATES:
    row = {"model": model_id}
    try:
        info = model_info(model_id)
        row["license"] = (info.card_data.license if info.card_data else None) or "?"
        row["revision"] = info.sha
        t = time.time()
        llm = LLMInterface(ModelConfig(model_id=model_id, revision=info.sha)).load_model()
        row["load_s"] = round(time.time() - t)
        row["gpu_gib"] = round(torch.cuda.memory_allocated() / 1024**3, 1)

        cache, f1s, invalid = PredictionCache(), [], 0
        t = time.time()
        for name, p in get_baseline_prompts():
            r = evaluate_prompt(p, sample.text, sample.label, llm.predict_raw_batch, model_id=llm.model_id,
                                inference_config=llm.inference_config, cache=cache, batch_size=100, split_name="opt")
            f1s.append(r["metrics"]["f1_macro"])
            invalid += r["metrics"]["n_invalid"]
        secs = time.time() - t
        row.update(mean_f1=round(sum(f1s) / len(f1s), 4), best_f1=round(max(f1s), 4), worst_f1=round(min(f1s), 4),
                   invalid_answers=invalid, reviews_per_s=round(8 * len(sample) / secs))

        # can this model also write usable prompts (it is the prompt writer too)?
        llm.set_seed(42)
        rng, calls0 = random.Random(42), llm.stats["generate_calls"]
        try:
            generate_initial_prompts(llm, 10, seed_prompts=get_baseline_prompt_texts()[:4], rng=rng)
            row["llm_calls_for_6_new_prompts"] = llm.stats["generate_calls"] - calls0
        except RuntimeError as exc:
            row["llm_calls_for_6_new_prompts"] = f"FAILED: {str(exc)[:70]}"
        seeds = get_baseline_prompt_texts()
        row["mutation_fallbacks_of_6"] = sum(mutate_prompt(llm, seeds[i], rng=rng) == seeds[i] for i in range(6))
    except Exception as exc:  # one model failing must not stop the others
        row["error"] = f"{type(exc).__name__}: {str(exc)[:200]}"
        traceback.print_exc()
    rows.append(row)
    llm = None
    gc.collect()
    torch.cuda.empty_cache()

comparison = pd.DataFrame(rows)
print(comparison.T.to_string())
comparison.to_csv("/kaggle/working/outputs/model_comparison.csv", index=False)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

mutate_prompt: no valid mutation after 3 attempts; returning parent


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (16 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

                                                                    0                                         1                                         2
model                                      Qwen/Qwen2.5-1.5B-Instruct                  Qwen/Qwen2.5-3B-Instruct           microsoft/Phi-3.5-mini-instruct
license                                                    apache-2.0                                     other                                       mit
revision                     989aa7980e4cf806f80c7fef2b1adb7bc71aa306  aa8e72537993ba99e69dfaafa59ed015b17504d1  2fe192450127e6a83f7441aef6e3ca586c338b77
load_s                                                              9                                        43                                        61
gpu_gib                                                           2.9                                       5.9                                       7.1
mean_f1                                                        0.9258       